# Qwen3.5-4B RotQuant K/V cache research matrix

This notebook measures rotation-aware K/V cache quality at the true post-RoPE Transformers cache boundary. It loads and packs the 4-bit RotQuant weight model once per seed, then runs a broad cache matrix without repeatedly rebuilding the model. Every completed trial is content-addressed and persisted to Google Drive.

## Goal

Find the cache quality/size Pareto frontier and test whether held-out dynamic allocation beats uniform precision at the same exact byte budget. The matrix includes all 2/3/4/8-bit K/V pairs, codebook and metadata ablations, structured-rotation block sizes, six dynamic budgets, source-weight controls, long-context confirmation, and three-seed validation.

### Key assumptions

- Dynamic selection and final evaluation use disjoint C4 sequences. Long-context confirmation uses a later skip range.
- K/V is rotated and quantized after RoPE. Recurrent linear-attention state remains unchanged and is reported separately.
- CUDA `fallback=true` accelerates the already-quantized **weights** and invalidates packed-weight memory/throughput claims. K/V bytes are still exact logical packed bytes.
- This is a quality and rate study. Native fused-cache latency remains a separate llama.cpp/CUDA benchmark.
- WikiText-2 is not used to select a cache recipe.

## Setup

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/CodeHalwell/rotquant.git"
REPO_REF = "main"
REPO_DIR = Path("/content/rotquant-kv-matrix")
MODEL_ID = "unsloth/Qwen3.5-4B"
CONFIG_RELATIVE_PATH = Path("configs/qwen35_4b_dynamic_kv_cuda.yaml")

USE_GOOGLE_DRIVE = True
DRIVE_RESULT_ROOT = Path("/content/drive/MyDrive/rotquant/qwen35_kv_matrix")
LOCAL_RESULT_ROOT = Path("/content/qwen35_kv_matrix")

BITS = (2, 3, 4, 8)
CODEBOOKS = ("gaussian", "uniform", "nf")
GROUP_SIZES = (32, 64, 128)
ROTATION_BLOCKS = (64, 128, 256)
DYNAMIC_BUDGETS = (2.25, 3.25, 4.25, 5.25, 6.25, 8.25)

DEV_BATCHES = 4
DEV_SELECTION_BATCHES = 4
DEV_PROMPT_LEN = 256
DEV_CONTINUATION_LEN = 16
DEV_SKIP = 384
LONG_BATCHES = 4
LONG_SELECTION_BATCHES = 4
LONG_PROMPT_LEN = 1024
LONG_CONTINUATION_LEN = 32
LONG_SKIP = 1024

CONFIRM_EXPENSIVE_RUN = False  # Inspect the matrix, then set True.
FORCE_RERUN = False
RUN_SOURCE_WEIGHT_CONTROLS = True
RUN_FULL_MATRIX = True
RUN_AGGRESSIVE_DYNAMIC_BUDGETS = True
RUN_SEED_VALIDATION = True
RUN_LONG_CONTEXT_CONFIRMATION = True
DOWNLOAD_RESULTS = True

print({"repo_ref": REPO_REF, "bits": BITS, "dynamic_budgets": DYNAMIC_BUDGETS})

### 1. Verify the RTX 6000-class runtime

In [ ]:
import os
import subprocess
import sys
import torch

assert torch.cuda.is_available(), "Select a CUDA GPU runtime before continuing."
gpu = torch.cuda.get_device_properties(0)
vram_gib = gpu.total_memory / 2**30
print(f"GPU: {gpu.name} | VRAM: {vram_gib:.1f} GiB | torch={torch.__version__} | CUDA={torch.version.cuda}")
if vram_gib < 40:
    print("WARNING: the cached fp16 weight fallback may OOM below 40 GiB.")
subprocess.run(["nvidia-smi"], check=True)

### 2. Mount persistence, fetch the exact revision, and install dependencies

In [ ]:
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULT_BASE = DRIVE_RESULT_ROOT
else:
    RESULT_BASE = LOCAL_RESULT_ROOT
RESULT_BASE.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", REPO_REF, "--single-branch", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", REPO_REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", REPO_REF], cwd=REPO_DIR, check=True)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
RESULT_ROOT = RESULT_BASE / commit[:12]
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Using commit {commit}; results: {RESULT_ROOT}")

In [ ]:
runtime_packages = [
    "transformers>=5.9,<6", "datasets>=4.8", "accelerate",
    "safetensors", "sentencepiece", "scipy", "pyyaml",
    "pandas", "matplotlib", "huggingface_hub",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *runtime_packages], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR), "--no-deps"], check=True)
pil_probe_command = [
    sys.executable, "-c",
    "from PIL import Image, ImageColor, ImageDraw, ImageFont, ImageText; print(Image.__version__)",
]
pil_probe = subprocess.run(pil_probe_command, capture_output=True, text=True)
if pil_probe.returncode != 0:
    print("Detected an inconsistent live Pillow installation; repairing it once.")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-cache-dir", "pillow==12.2.0"],
        check=True,
    )
    subprocess.run(pil_probe_command, check=True)
    raise RuntimeError(
        "Pillow was repaired. Use Runtime > Restart session, then rerun the notebook from the top."
    )
os.environ["TORCH_ALLOW_TF32_CUBLAS_OVERRIDE"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")
print(f"Runtime installed without replacing CUDA PyTorch; Pillow={pil_probe.stdout.strip()}.")

### 3. Validate the repository contract

In [ ]:
import json
import yaml

config_path = REPO_DIR / CONFIG_RELATIVE_PATH
required_paths = [config_path, REPO_DIR / "eval/kv_cache.py", REPO_DIR / "rotquant/kv_cache.py"]
missing = [str(path) for path in required_paths if not path.exists()]
assert not missing, "Missing required files: " + ", ".join(missing)
with config_path.open() as handle:
    experiment_config = yaml.safe_load(handle)
assert experiment_config["model"] == MODEL_ID
assert experiment_config["patch"]["fallback"] is True
source = (REPO_DIR / "eval/kv_cache.py").read_text()
assert "select_dynamic_kv_quantization" in source
assert "source_kv_elements" in source
print(yaml.safe_dump(experiment_config, sort_keys=False))

## Steps

### 4. Build the complete profile matrix

In [ ]:
from copy import deepcopy

BASE_KV = {
    "bits": 4, "group_size": 64, "codebook": "gaussian",
    "rotation_block": 128, "batches": DEV_BATCHES,
    "eval_offset_batches": DEV_SELECTION_BATCHES,
    "prompt_len": DEV_PROMPT_LEN, "continuation_len": DEV_CONTINUATION_LEN,
    "skip": DEV_SKIP,
}

def uniform_profile(key_bits, value_bits, **changes):
    profile = {**BASE_KV, "key_bits": key_bits, "value_bits": value_bits}
    profile.update(changes)
    return profile

def dynamic_profile(target_bpv, **changes):
    profile = dict(BASE_KV)
    profile["dynamic"] = {
        "candidate_bits": list(BITS), "target_bpv": target_bpv,
        "selection_batches": DEV_SELECTION_BATCHES,
    }
    profile.update(changes)
    return profile

profiles = {}
for key_bits in BITS:
    for value_bits in BITS:
        profiles[f"uniform_k{key_bits}_v{value_bits}"] = uniform_profile(key_bits, value_bits)

for codebook in CODEBOOKS:
    for group_size in GROUP_SIZES:
        if codebook == "gaussian" and group_size == 64:
            continue
        profiles[f"codebook_{codebook}_g{group_size}"] = uniform_profile(4, 4, codebook=codebook, group_size=group_size)

for rotation_block in ROTATION_BLOCKS:
    if rotation_block == 128:
        continue
    profiles[f"rotation_block_{rotation_block}"] = uniform_profile(4, 4, rotation_block=rotation_block)

for target_bpv in DYNAMIC_BUDGETS:
    if not RUN_AGGRESSIVE_DYNAMIC_BUDGETS and target_bpv < 4.25:
        continue
    profiles[f"dynamic_{target_bpv:.2f}bpv"] = dynamic_profile(target_bpv)

if not RUN_FULL_MATRIX:
    keep = {"uniform_k3_v3", "uniform_k3_v4", "uniform_k4_v3", "uniform_k4_v4", "dynamic_3.25bpv", "dynamic_4.25bpv"}
    profiles = {name: value for name, value in profiles.items() if name in keep}

print(f"Seed-0 packed-weight profiles: {len(profiles)}")
print("\n".join(profiles))

### 5. Define a one-load-per-seed, resumable evaluator

In [ ]:
import gc
import hashlib
import time
from dataclasses import asdict

sys.path.insert(0, str(REPO_DIR))
from eval.kv_cache import KVDynamicConfig, KVCacheEvalConfig, evaluate_kv_cache
from rotquant.patch import PatchConfig, patch_model
from rotquant.quantize import QuantConfig
from rotquant.utils import environment_record, set_seed
from scripts.run_experiment import build_calib_loader, footprint_metrics, load_hf_model

trial_records = {}
batch_cache = {}

def _trial_path(name, seed, source_weights, profile):
    signature = hashlib.sha256(json.dumps({
        "commit": commit, "name": name, "seed": seed,
        "source_weights": source_weights, "profile": profile,
    }, sort_keys=True).encode()).hexdigest()[:12]
    return RESULT_ROOT / f"{name}_s{seed}_{signature}.json"

def _batches(tokenizer, config):
    selection = config.eval_offset_batches
    if config.dynamic:
        selection = max(selection, KVDynamicConfig(**config.dynamic).selection_batches)
    count = config.batches + selection
    seq_len = config.prompt_len + config.continuation_len + 1
    key = (count, seq_len, config.skip)
    if key not in batch_cache:
        batch_cache[key] = build_calib_loader(tokenizer, count, seq_len, "cuda", skip=config.skip)
    return batch_cache[key]

def evaluate_trial(model, tokenizer, name, profile, seed, source_weights, weight_metrics):
    profile = deepcopy(profile)
    profile["seed"] = seed
    output_path = _trial_path(name, seed, source_weights, profile)
    if output_path.exists() and not FORCE_RERUN:
        with output_path.open() as handle:
            payload = json.load(handle)
        print(f"Reusing {output_path.name}")
    else:
        config = KVCacheEvalConfig(**profile)
        started = time.perf_counter()
        metrics = evaluate_kv_cache(model, _batches(tokenizer, config), config, "cuda")
        metrics["seconds"] = time.perf_counter() - started
        payload = {
            "trial": name, "seed": seed, "source_weights": source_weights,
            "model": MODEL_ID, "git_sha": commit, "kv_config": asdict(config),
            "weight_metrics": weight_metrics, "metrics": metrics,
            "environment": environment_record(),
        }
        with output_path.open("w") as handle:
            json.dump(payload, handle, indent=2)
        print(f"Wrote {output_path.name}: KL={metrics['mean_teacher_kl']:.4g}, bpv={metrics['effective_kv_bpv']:.3f}")
    trial_records[(name, seed, source_weights)] = payload
    return payload

def load_model(seed):
    set_seed(seed)
    model, tokenizer, loader = load_hf_model(MODEL_ID, torch.float16, "cuda", "multimodal_lm")
    model.eval()
    return model, tokenizer

def patch_weights(model, seed):
    quant = QuantConfig(bits=4, codebook="gaussian", scale="mse_search", group_size=128, error_comp="none", seed=seed)
    patch = PatchConfig(
        quant=quant, rotation="fwht", block=128, mode="consistent",
        fallback=True, seed=seed, include=["model.language_model.layers."],
        exclude=["linear_attn.in_proj_a", "linear_attn.in_proj_b"],
    )
    started = time.perf_counter()
    patch_model(model, patch)
    metrics = footprint_metrics(model, {})
    metrics["patch_seconds"] = time.perf_counter() - started
    return metrics

def release_cuda_memory():
    gc.collect()
    torch.cuda.empty_cache()

### 6. Confirm the full matrix

Set `CONFIRM_EXPENSIVE_RUN=True` in the parameter cell after checking the GPU and Drive path.

In [ ]:
assert CONFIRM_EXPENSIVE_RUN, "Set CONFIRM_EXPENSIVE_RUN=True before launching the matrix."

### 7. Run source-weight controls, then the complete seed-0 RotQuant matrix

In [ ]:
model, tokenizer = load_model(seed=0)
if RUN_SOURCE_WEIGHT_CONTROLS:
    evaluate_trial(model, tokenizer, "source_uniform_k4_v4", uniform_profile(4, 4), 0, True, {})
    evaluate_trial(model, tokenizer, "source_dynamic_4.25bpv", dynamic_profile(4.25), 0, True, {})
weight_metrics_seed0 = patch_weights(model, seed=0)
print(json.dumps(weight_metrics_seed0, indent=2))
for profile_name, profile in profiles.items():
    evaluate_trial(model, tokenizer, profile_name, profile, 0, False, weight_metrics_seed0)
model = None
release_cuda_memory()

## Checks

### 8. Build the seed-0 table and Pareto frontier

In [ ]:
import numpy as np
import pandas as pd

def report_row(payload):
    metrics = payload["metrics"]
    dynamic = metrics.get("dynamic", {})
    return {
        "trial": payload["trial"], "seed": payload["seed"],
        "source_weights": payload["source_weights"],
        "teacher_kl": metrics["mean_teacher_kl"],
        "logit_cosine": metrics["mean_logit_cosine"],
        "top1": metrics["top1_agreement"], "nll_delta": metrics["nll_delta"],
        "key_bits": metrics["key_bits"], "value_bits": metrics["value_bits"],
        "effective_bpv": metrics["effective_kv_bpv"],
        "packed_kv_MB": metrics["packed_kv_bytes"] / 1e6,
        "compression": metrics["kv_compression_ratio"],
        "seconds": metrics["seconds"],
        "restored_uniform": dynamic.get("restored_uniform", False),
        "target_reached": dynamic.get("target_reached", True),
    }

seed0_payloads = [payload for (_, seed, source), payload in trial_records.items() if seed == 0 and not source]
seed0 = pd.DataFrame([report_row(payload) for payload in seed0_payloads]).sort_values(["packed_kv_MB", "teacher_kl"])
seed0["score"] = seed0["teacher_kl"] + 0.01 * seed0["nll_delta"].clip(lower=0)

def pareto_mask(frame):
    rows = frame.reset_index(drop=True)
    keep = []
    for index, row in rows.iterrows():
        dominated = ((rows["packed_kv_MB"] <= row["packed_kv_MB"]) & (rows["score"] <= row["score"]) & ((rows["packed_kv_MB"] < row["packed_kv_MB"]) | (rows["score"] < row["score"]))).any()
        keep.append(not dominated)
    return np.array(keep, dtype=bool)

pareto = seed0.reset_index(drop=True)[pareto_mask(seed0)]
display(seed0.style.format({"teacher_kl": "{:.3g}", "nll_delta": "{:+.4f}", "effective_bpv": "{:.3f}", "packed_kv_MB": "{:.3f}", "compression": "{:.2f}x"}))
print("Pareto profiles:", pareto["trial"].tolist())

### 9. Select equal-budget winners and validate the Pareto set across seeds

In [ ]:
reference4 = seed0.loc[seed0["trial"] == "uniform_k4_v4"].iloc[0]
reference3 = seed0.loc[seed0["trial"] == "uniform_k3_v3"].iloc[0]
winner4 = seed0[seed0["packed_kv_MB"] <= reference4["packed_kv_MB"] + 1e-9].sort_values("score").iloc[0]
winner3 = seed0[seed0["packed_kv_MB"] <= reference3["packed_kv_MB"] + 1e-9].sort_values("score").iloc[0]
validation_names = list(dict.fromkeys([*pareto["trial"].tolist(), "uniform_k3_v3", "uniform_k4_v4", winner3["trial"], winner4["trial"]]))
print({"winner_at_3bit_budget": winner3["trial"], "winner_at_4bit_budget": winner4["trial"], "seed_validation_profiles": validation_names})

if RUN_SEED_VALIDATION:
    for seed in (1, 2):
        model, tokenizer = load_model(seed)
        weight_metrics = patch_weights(model, seed)
        for profile_name in validation_names:
            evaluate_trial(model, tokenizer, profile_name, profiles[profile_name], seed, False, weight_metrics)
        model = None
        release_cuda_memory()
else:
    print("Seed validation disabled; do not treat seed-0 rankings as final.")

### 10. Confirm the 3-bit- and 4-bit-budget winners at 1,024-token context

In [ ]:
def long_profile(profile):
    result = deepcopy(profile)
    result.update({"batches": LONG_BATCHES, "eval_offset_batches": LONG_SELECTION_BATCHES, "prompt_len": LONG_PROMPT_LEN, "continuation_len": LONG_CONTINUATION_LEN, "skip": LONG_SKIP})
    if result.get("dynamic"):
        result["dynamic"] = dict(result["dynamic"], selection_batches=LONG_SELECTION_BATCHES)
    return result

if RUN_LONG_CONTEXT_CONFIRMATION:
    model, tokenizer = load_model(0)
    weight_metrics = patch_weights(model, 0)
    long_names = list(dict.fromkeys([winner3["trial"], "uniform_k3_v3", winner4["trial"], "uniform_k4_v4"]))
    for profile_name in long_names:
        evaluate_trial(model, tokenizer, f"long_{profile_name}", long_profile(profiles[profile_name]), 0, False, weight_metrics)
    model = None
    release_cuda_memory()
else:
    print("Long-context confirmation disabled.")

## Results

### 11. Write the complete report, figures, and durable experiment entry

In [ ]:
all_rows = pd.DataFrame([report_row(payload) for payload in trial_records.values()])
all_rows["score"] = all_rows["teacher_kl"] + 0.01 * all_rows["nll_delta"].clip(lower=0)
report_path = RESULT_ROOT / "kv_trial_matrix.csv"
all_rows.sort_values(["source_weights", "seed", "packed_kv_MB", "score"]).to_csv(report_path, index=False)
display(all_rows.sort_values(["seed", "packed_kv_MB", "score"]).style.format({"teacher_kl": "{:.3g}", "nll_delta": "{:+.4f}", "effective_bpv": "{:.3f}", "packed_kv_MB": "{:.3f}"}))
print(f"Wrote {report_path}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 6))
seed0_plot = all_rows[(all_rows["seed"] == 0) & (~all_rows["source_weights"]) & (~all_rows["trial"].str.startswith("long_"))]
ax.scatter(seed0_plot["effective_bpv"], seed0_plot["teacher_kl"], alpha=0.65, label="all seed-0 profiles")
for _, row in pareto.iterrows():
    ax.scatter(row["effective_bpv"], row["teacher_kl"], s=90, color="tab:red")
    ax.annotate(row["trial"], (row["effective_bpv"], row["teacher_kl"]), xytext=(5, 5), textcoords="offset points", fontsize=8)
ax.set_yscale("log")
ax.set_xlabel("Exact effective cache bits/value (codes + scales + padding)")
ax.set_ylabel("Mean teacher KL (log scale, lower is better)")
ax.set_title("Qwen3.5-4B RotQuant K/V cache rate-distortion frontier")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plot_path = RESULT_ROOT / "kv_rate_distortion_frontier.png"
fig.savefig(plot_path, dpi=180)
plt.show()
print(f"Wrote {plot_path}")

In [ ]:
summary = {
    "git_sha": commit, "model": MODEL_ID,
    "seed0_profiles": len(seed0), "pareto_profiles": pareto["trial"].tolist(),
    "winner_at_3bit_budget": winner3["trial"],
    "winner_at_4bit_budget": winner4["trial"],
    "seeds_completed": sorted(all_rows["seed"].unique().tolist()),
    "long_context_completed": bool(all_rows["trial"].str.startswith("long_").any()),
}
with (RESULT_ROOT / "kv_summary.json").open("w") as handle:
    json.dump(summary, handle, indent=2)

ledger = f"""## External Qwen3.5-4B K/V matrix\n\n- Git SHA: `{commit}`\n- GPU: {gpu.name} ({vram_gib:.1f} GiB)\n- Seed-0 profiles: {len(seed0)}\n- Pareto profiles: {', '.join(pareto['trial'].tolist())}\n- 3-bit-budget winner: `{winner3['trial']}`\n- 4-bit-budget winner: `{winner4['trial']}`\n- Raw table: `kv_trial_matrix.csv`\n- Plot: `kv_rate_distortion_frontier.png`\n\nInterpretation must be completed after checking multi-seed and long-context rows. CUDA weight fallback was quality-only; cache byte accounting was exact.\n"""
ledger_path = RESULT_ROOT / "experiment_log_entry.md"
ledger_path.write_text(ledger)
print(json.dumps(summary, indent=2))
print(f"Wrote {ledger_path}")

### 12. Archive the compact record

In [ ]:
import shutil

archive_base = Path("/content/qwen35_kv_cache_matrix")
archive_path = Path(shutil.make_archive(str(archive_base), "zip", RESULT_ROOT))
print(f"Created {archive_path} ({archive_path.stat().st_size / 1e6:.2f} MB)")
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(str(archive_path))

## Takeaways

After execution, use `kv_trial_matrix.csv` rather than notebook scrollback as the source of truth. A dynamic recipe is a win only if it beats the matched uniform recipe at no more exact bytes across seeds and survives the disjoint 1,024-token confirmation. Send back the CSV, summary JSON, and experiment-log entry so the repository ledger can be updated without transcription errors.